In [1]:
import pandas as pd
import os

assessments         = pd.read_csv("cleaned/assessments.csv")
courses             = pd.read_csv("cleaned/courses.csv")
studentAssessment   = pd.read_csv("cleaned/studentAssessment.csv")
studentRegistration = pd.read_csv("cleaned/studentRegistration.csv")
studentInfo         = pd.read_csv("cleaned/studentInfo.csv")
dim_enrollment      = pd.read_csv("cleaned/dim_enrollment.csv")
studentVle          = pd.read_csv("cleaned/studentVle.csv")
vle                 = pd.read_csv("cleaned/vle.csv")

print("All tables loaded")

All tables loaded


In [2]:
#sql server connection
import sqlalchemy as sal
engine = sal.create_engine("mssql+pyodbc://SONU/master?driver=ODBC Driver 17 for SQL Server")
conn = engine.connect()

In [3]:
from sqlalchemy import VARCHAR,Integer,Numeric

def get_dtypes(df):
    dtypes = {}

    for col in df.columns:
        if df[col].dtype == "object":
            max_len = df[col].str.len().max()
            dtypes[col] = VARCHAR(int(max_len)+20)
        elif df[col].dtype in ["int64","int32"]:
            dtypes[col] = Integer()
        elif df[col].dtype in ["float64", "float32"]:
            dtypes[col] = Numeric(10, 2)
    return dtypes

In [4]:
tables = [
    ("dim_course", courses),
    ("dim_vle", vle),
    ("dim_assessment", assessments),
    ("dim_student", studentInfo),
    ("dim_enrollment", dim_enrollment),
    ("dim_registration", studentRegistration),
    ("fact_activity", studentVle),
    ("fact_assessment", studentAssessment)
]

for table_name, df in tables:
    chunk = 5000 if len(df) > 10000 else None
    try:
        df.to_sql(table_name,con=conn,index=False,if_exists="replace",chunksize=chunk,dtype=get_dtypes(df))
        print(f"Table '{table_name}' created successfully.")
    except Exception as e:
        print(f"Error creating table '{table_name}': {e}")

Table 'dim_course' created successfully.
Table 'dim_vle' created successfully.
Table 'dim_assessment' created successfully.
Table 'dim_student' created successfully.
Table 'dim_enrollment' created successfully.
Table 'dim_registration' created successfully.
Table 'fact_activity' created successfully.
Table 'fact_assessment' created successfully.
